# StarTrek Reproducibility Analysis

This notebook allows verifying the reproducibility of training results using 5 different seeds. It generates a statistical report including the mean and the 95% confidence interval.

## 1. Imports and Configuration

# Libraries used for this test file.

In [ ]:
import subprocess
import re
import math
import matplotlib.pyplot as plt
import sys
import os
import time

# Configuration
- `SEEDS` = the worlds where the robot will train
- `CONFIG` = path to the test configuration file
- `TRAIN_SCRIPT` = path to the training script
- `NP_EP_DEFAULT` = default number of episodes

In [ ]:
SEEDS = [89, 67, 69, 62, 99]
CONFIG = "configs/exp_fast.yaml"
TRAIN_SCRIPT = "train.py"
NP_EP_DEFAULT = 400

## Execution: number of episodes
`get_total_episodes` returns either the number of episodes defined in the `.yaml` configuration file (`CONFIG`), or the `NP_EP_DEFAULT` value.

## Recommendation
It is recommended that `NP_EP_DEFAULT` be between **400** and **600** to obtain sufficient training and reliable final results.

In [ ]:
def get_total_episodes(config_path):
    try:
        path = f"../{config_path}" if not os.path.exists(config_path) else config_path
        with open(path, "r") as f:
            content = f.read()
            match = re.search(r"episodes:\s*(\d+)", content)
            if match:
                return int(match.group(1))
    except Exception:
        pass
    return NP_EP_DEFAULT

# Launching Training
The training script is launched as a subprocess to calculate the estimated time remaining (**ETA**).
Each training is executed for each configured `SEED` and its score is collected at the end to calculate the average.

# Note
Training time can be quite long depending on the number of episodes configured in `NP_EP_DEFAULT`. The ETA provides an estimation of the remaining time.

In [ ]:

def run_training(seed):
    print(f"[*] Starting training with seed {seed}...")
    total_episodes = get_total_episodes(CONFIG)
    
    process = subprocess.Popen(
        [sys.executable, TRAIN_SCRIPT, CONFIG, str(seed)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="..",
        bufsize=1
    )
    
    scores = []
    start_time = time.time()
    
    for line in iter(process.stdout.readline, ""):
        line = line.strip()
        if not line:
            continue
            
        print(line)
        match = re.search(r"Score:\s*([\d\.-]+)", line)
        if match:
            try:
                scores.append(float(match.group(1)))
                # Calculating and displaying ETA for the current seed
                episodes_done = len(scores)
                elapsed = time.time() - start_time
                avg_time_per_ep = elapsed / episodes_done
                episodes_left = total_episodes - episodes_done
                eta_seconds = max(0, avg_time_per_ep * episodes_left)
                
                eta_str = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))
                print(f"    -> [ETA for the end of seed {seed}: {eta_str}]")
            except Exception:
                pass

    process.stdout.close()
    process.wait()

    if process.returncode != 0:
        print(f"[!] Training failed with code {process.returncode}")
        
    return scores

## 2. Running Training Sessions

This step can take several minutes depending on the number of configured episodes.
We add a system that calculates an estimate of the completion time for the series (ETA).

The following program simply serves to calculate the ETA to estimate the time required to complete all series.

In [ ]:
import time

results = {}
start_time_total = time.time()

for i, seed in enumerate(SEEDS):
    start_time_seed = time.time()
    results[seed] = run_training(seed)
    end_time_seed = time.time()
    
    # Time calculations
    elapsed_seed = end_time_seed - start_time_seed
    elapsed_total = end_time_seed - start_time_total
    
    # Estimating remaining time
    seeds_done = i + 1
    seeds_left = len(SEEDS) - seeds_done
    avg_time_per_seed = elapsed_total / seeds_done
    eta_seconds = avg_time_per_seed * seeds_left
    
    eta_str = time.strftime("%H:%M:%S", time.gmtime(eta_seconds))
    print(f"[-] Seed {seed} completed in {elapsed_seed:.2f}s. ETA for remaining seeds: {eta_str}\n")


## 3. Statistical Analysis and Visualization

Calculation of the mean and confidence interval (Target: 95%)

# Chart

This chart shows the evolution of the average score over episodes, calculated from runs corresponding to different `SEEDS`.

- **Central curve**: average score at each episode.
- **Shaded area (95% CI)**: confidence interval centered on the mean to estimate variability between `SEEDS`.
- **Procedure**: all series are truncated to the same length (minimum of lengths) to compare step by step, calculating the mean and empirical standard deviation.

To estimate the final performance of each seed, we take the simple average of the **last 100 results** (or fewer if the run is shorter) to capture actual performance rather than the training phase.

In [ ]:
all_runs = [results[s] for s in SEEDS if len(results[s]) > 0]

if not all_runs:
    print("[!] Error: No scores collected. Check if training ran correctly.")
else:
    min_len = min(len(r) for r in all_runs)
    all_runs_trimmed = [r[:min_len] for r in all_runs]
    n = len(all_runs_trimmed)

    mean_curve = []
    for i in range(min_len):
        val_at_i = [run[i] for run in all_runs_trimmed]
        mean_curve.append(sum(val_at_i) / n)
    ci_curve = []
    t_value = 2.776
    if n < 5:
        t_map = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776}
        t_value = t_map.get(n-1, 2.0)
    for i in range(min_len):
        val_at_i = [run[i] for run in all_runs_trimmed]
        m = mean_curve[i]
        variance = sum((x - m)**2 for x in val_at_i) / (n - 1) if n > 1 else 0
        std_dev = math.sqrt(variance)
        std_error = std_dev / math.sqrt(n)
        ci_curve.append(t_value * std_error)

    plt.figure(figsize=(12, 6))
    episodes = list(range(min_len))
    plt.plot(episodes, mean_curve, color='#1f77b4', label='Average Score', linewidth=2)

    lower_bound = [m - c for m, c in zip(mean_curve, ci_curve)]
    upper_bound = [m + c for m, c in zip(mean_curve, ci_curve)]

    plt.fill_between(episodes, lower_bound, upper_bound, color='#1f77b4', alpha=0.2, label='95% CI')

    plt.title(f'StarTrek Performance (DQN) over {n} Seeds', fontsize=14, fontweight='bold')
    plt.xlabel('Episodes', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.legend(frameon=True, shadow=True)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

## 4. Final Performance Report

This section summarizes the final performance for each run and provides a reproducibility estimate:

- **Window used**: we take the average of the last 100 episodes of each run (or fewer if the run is shorter) to estimate stable final performance.
- **Final metric**: mean ± 95% CI calculated on the final performances of the runs.

Verification steps:
- If too much variance: increase the number of ``SEEDS`` or extend training to reduce standard error.

In [ ]:
if not all_runs:
    print("[!] No final report possible.")
else:
    final_performances = []
    for run in all_runs:
        # Taking the last 100 episodes
        window = min(100, len(run))
        last_window = run[-window:]
        if last_window:
            final_performances.append(sum(last_window) / len(last_window))

    if final_performances:
        n_final = len(final_performances)
        final_mean = sum(final_performances) / n_final
        final_variance = sum((x - final_mean)**2 for x in final_performances) / (n_final - 1) if n_final > 1 else 0
        final_std_error = math.sqrt(final_variance) / math.sqrt(n_final)
        
        t_map = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776}
        t_val = t_map.get(n_final-1, 2.0)
        final_ci = t_val * final_std_error

        print("="*40)
        print(" REPRODUCIBILITY REPORT ")
        print("="*40)
        print(f"Configuration: {CONFIG}")
        print(f"Successful runs: {n_final}/{len(SEEDS)}")
        print(f"Final Performance: {final_mean:.2f} ± {final_ci:.2f}")
        print("="*40)
    else:
        print("[!] Not enough data for the final report.")